# Validação de Tendência: IFR 200 como Divisor (Linha 50)

Este notebook tem como objetivo validar a hipótese de que o **IFR de 200 períodos** funciona como um divisor de tendência primário:
- **IFR 200 > 50**: Tendência de Alta (Bull Regime).
- **IFR 200 < 50**: Tendência de Baixa (Bear Regime).

Vamos analisar o comportamento do preço (retornos futuros) em cada um desses regimes.

In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Adicionar o diretório raiz ao path para importar as ferramentas do projeto
script_dir = os.getcwd() # Se estiver no VS Code, o CWD costuma ser a raiz ou a pasta do notebook
project_root = os.path.abspath(os.path.join(script_dir, "../../"))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.research.utils import get_data_path, calculate_rsi_200
from src.core.config import IFR_PERIOD

print(f"IFR Period Configurado: {IFR_PERIOD}")

## 1. Carregamento de Dados
Vamos utilizar o **WIN (Mini Índice)** no timeframe de **15 minutos** para esta validação inicial.

In [ ]:
symbol = "WIN$"
timeframe = "15"
path = get_data_path(symbol, timeframe)

# Carregar usando Polars para performance
df_pl = pl.read_parquet(path)
print(f"Linhas carregadas: {len(df_pl)}")

# Converter para Pandas para análise estatística e plotagem
df = df_pl.to_pandas()
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time')
df.head()

## 2. Cálculo do IFR 200
Aplicamos o cálculo de Wilder (EWM) com 200 períodos.

In [ ]:
df['ifr_200'] = calculate_rsi_200(df['close'].values, period=200)

# Definir Regimes
df['regime'] = np.where(df['ifr_200'] > 50, 'BULL', 'BEAR')
df.loc[df['ifr_200'].isna(), 'regime'] = np.nan

print("Distribuição de Regimes:")
print(df['regime'].value_counts(normalize=True) * 100)

## 3. Análise de Retornos Futuros
Vamos calcular o retorno esperado para os próximos 5, 10 e 20 candles em cada regime.

In [ ]:
horizons = [5, 10, 20, 50]

for h in horizons:
    # Retorno percentual futuro
    df[f'ret_{h}'] = (df['close'].shift(-h) / df['close'] - 1) * 100

# Agrupar por regime e calcular métricas
metrics = ['mean', 'std', 'median']
results = df.groupby('regime')[[f'ret_{h}' for h in horizons]].agg(metrics)
results

## 4. Visualização de Performance Acumulada
Como seria a curva de capital se seguíssemos a tendência indicada pelo IFR 200?

In [ ]:
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

# Retorno da Estratégia: Long se Bull, Short se Bear
df['strategy_ret'] = df['log_ret'] * np.where(df['regime'].shift(1) == 'BULL', 1, -1)

plt.figure(figsize=(15, 7))
plt.plot(df['time'], df['log_ret'].cumsum(), label='Buy & Hold', color='gray', alpha=0.5)
plt.plot(df['time'], df['strategy_ret'].cumsum(), label='IFR 200 Trend Follow', color='blue')
plt.title(f"Equity Curve: IFR 200 como Divisor ({symbol} {timeframe}m)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Distribuição de Retornos (Histograma)
Existe um viés estatístico claro?

In [ ]:
plt.figure(figsize=(12, 6))
sns.kdeplot(df[df['regime'] == 'BULL']['ret_20'], label='Retornos em Bull (IFR > 50)', fill=True)
sns.kdeplot(df[df['regime'] == 'BEAR']['ret_20'], label='Retornos em Bear (IFR < 50)', fill=True)
plt.axvline(0, color='red', linestyle='--')
plt.title("Densidade de Retornos Futuros (20 candles) por Regime")
plt.xlabel("Retorno (%)")
plt.legend()
plt.show()

## 6. Conclusão Provisória
- Se a média dos retornos em BULL for positiva e em BEAR for negativa, a linha 50 é um divisor válido.
- Se a curva de tendência superar o Buy & Hold, o IFR 200 possui valor preditivo de momentum.